In [1]:
%pip install lifelines

Note: you may need to restart the kernel to use updated packages.


Phase 4: Survival Analysis Module

Builds Kaplan-Meier survival curves and Cox Proportional Hazards models per circuit, reframing tire degradation as a "probability of still being being competitive" question

In [2]:
import pandas as pd
import numpy as np
import os
import logging
import warnings 
import joblib

from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter, CoxPHFitter

warnings.filterwarnings('ignore')

SETUP

In [3]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'processed'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'), exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase1_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

CONFIGURATION

In [4]:
FAILURE_THRESHOLD = 1.03 #a lap 3% slowe than stint's best = "failed"
MIN_STINTS = 8   #minimum stints required to fit survival models

LOAD ENIGINEERED DATA From Previous Phases

In [5]:
def load_data():
    path = os.path.join(BASE, 'data', 'processed', 'engineered_laps.csv')
    df = pd.read_csv(path)
    log.info(f"Loaded engineered_laps.csv: {len(df)} rows")
    return df

BUILD STINT-LEVEL SURVIVAL DATASET

Converts lap-level data into one row per stint, with a 'duration' (tire age at a failure or consoring) and 'event' flag 

(TRUE = genuinely failed, FALSE = censored by an earlier pit stop)

In [6]:
def build_stint_dataset(df):
    log.info("BUILDING STINT-LEVEL SIRVIVAL DATASET")
    log.info('-' * 50)

    rows = []

    group_cols = ['CircuitName', 'Driver', 'Stint', 'Compound']

    for (circuit, driver, stint, compound), group in df.groupby(group_cols):
        group = group.sort_values('TyreLife')

        if len(group) < 3:
            #Too short a stint to meaningfully judge the failure
            continue

        stint_best = group['LapTimeSeconds'].min()
        threshold = stint_best * FAILURE_THRESHOLD

        #Finding the first lap where lap time crosses the threshold
        failed_laps = group[group['LapTimeSeconds'] > threshold]

        if not failed_laps.empty:
            #Tire genuinely "failed" - take the tire age at that point
            duration = failed_laps.iloc[0]['TyreLife']
            event = True
        else:
            '''Never crossed the threshold - censored at the last lap
            of the stint (driver pitted while tire was still fine)'''
            duration = group['TyreLife'].max()
            event = False

        '''Grab average conditions for this stint - used as
        covariates in the Cox model later'''
        avg_track_temp = group['TrackTemp'].mean()
        avg_fuel_corrected = group['FuelCorrectedLapTime'].mean()

        rows.append({
            'CircuitName': circuit,
            'Driver': driver,
            'Stint': stint,
            'Compound': compound,
            'Duration': duration,
            'Event': event,
            'AvgTrackTemp': avg_track_temp,
            'AvgFuelCorrectedLapTime': avg_fuel_corrected,
            'CompoundEncoded': group['CompoundEncoded'].iloc[0]
                if 'CompoundEncoded' in group.columns else None
        })

    stint_df = pd.DataFrame(rows)

    log.info(f"Stint-level dataset built: {len(stint_df)} stints")
    log.info(f" Failure events: {stint_df['Event'].sum()}")
    log.info(f" Censored (pitted before failing):"
            f"{(~stint_df['Event']).sum()}")

    return stint_df

FIT KAPLAN MEIER PER CIRCUIT + COMPOUND

Produces a survival curve (with confidence bands) for every circuit+compound combination

In [7]:
def fit_kaplan_meier(stint_df):
    log.info("FITTING KAPLAN-MEIER SURVIVAL CURVES")
    log.info("-" * 50)

    curve_rows = []
    km_models = {}

    groups = stint_df.groupby(['CircuitName', 'Compound'])

    for (circuit, compound), group in groups:
        if len(group) < MIN_STINTS:
            log.warning(f" Skipping {circuit} / {compound} - "
                        f"only {len(group)} stints (need {MIN_STINTS}+)")
            continue

        kmf = KaplanMeierFitter()
        kmf.fit(
            durations = group['Duration'],
            event_observed = group['Event'],
            label = f"{circuit}_{compound}"
        )

        '''Extract the survival function and confidence intervals
        as a clean table - one row per tire age, ready for plotting'''
        survival_table = kmf.survival_function_.reset_index()
        survival_table.columns = ['TyreAge', 'SurvivalProbability']

        ci_table = kmf.confidence_interval_.reset_index()
        ci_table.columns = ['TyreAge', 'CI_Lower', 'CI_Upper']

        merged = survival_table.merge(ci_table, on='TyreAge')
        merged['CircuitName'] = circuit
        merged['Compound'] = compound

        curve_rows.append(merged)

        '''Save the fitted model itself for later use (e.g.
        dashboard can re-query survival probability at any exact
        tire age)'''
        key = f"{circuit}_{compound}"
        km_models[key] = kmf

        median_survival = kmf.median_survival_time_
        log.info(f" {circuit} / {compound}: {len(group)} stints |"
                f" median survival = {median_survival} laps")

    survival_curves_df = pd.concat(curve_rows, ignore_index=True)

    return survival_curves_df, km_models

FIT COX PROPORTIONAL HAZARDS PER CIRCUIT

Estimates how much each factor (tire age via duration,
compound, track-temp, fuel-corrected pace) increases or
decreases the insrantaneous risk of tire failure

In [8]:
def fit_cox_models(stint_df):
    log.info("=" * 50)
    log.info("FITTING COX PROPORTIONAL HAZARDS MODELS")
    log.info("=" * 50)

    hazard_rows = []
    cox_models = {}

    covariates = ['CompoundEncoded', 'AvgTrackTemp', 'AvgFuelCorrectedLapTime']

    for circuit, group in stint_df.groupby('CircuitName'):
        model_data = group[['Duration', 'Event'] + covariates].dropna()

        if len(model_data) < MIN_STINTS * 2:
            log.warning(f"  Skipping {circuit} — "
                        f"only {len(model_data)} usable stints")
            continue

        # Event column needs to be int (1/0) for lifelines
        model_data = model_data.copy()
        model_data['Event'] = model_data['Event'].astype(int)

        # Scale covariates so they're on comparable ranges.
        # Cox regression is numerically sensitive — without this,
        # mixing raw lap-time-scale values (~90-100) with encoded
        # compound values (0/1/2) and temperature (~30-50) commonly
        # causes convergence failures.
        scaler = StandardScaler()
        model_data[covariates] = scaler.fit_transform(model_data[covariates])

        # Drop rows with zero variance issues — if a circuit only
        # ever used one compound, CompoundEncoded will be constant
        # after scaling (all zeros), which also breaks Cox fitting
        if model_data[covariates].std().min() == 0:
            log.warning(f"  Skipping {circuit} — "
                        f"a covariate has zero variance")
            continue

        try:
            # penalizer adds a small regularization term (like Ridge)
            # which helps the model converge on real-world data
            # that doesn't perfectly separate
            cph = CoxPHFitter(penalizer=0.1)
            cph.fit(
                model_data,
                duration_col='Duration',
                event_col='Event'
            )

            # Extract hazard ratios (exp(coefficient)) per covariate
            summary = cph.summary.reset_index()
            summary['CircuitName'] = circuit
            summary = summary.rename(columns={
                'covariate': 'Covariate',
                'exp(coef)': 'HazardRatio',
                'p': 'PValue'
            })

            hazard_rows.append(
                summary[['CircuitName', 'Covariate', 'HazardRatio', 'PValue']]
            )

            cox_models[circuit] = cph

            log.info(f"  {circuit}: Cox model fit on {len(model_data)} stints")

        except Exception as e:
            log.error(f"  {circuit}: Cox model failed — {e}")
            continue

    if not hazard_rows:
        log.error("No Cox models were successfully fit for any circuit.")
        hazard_df = pd.DataFrame(columns=['CircuitName', 'Covariate', 'HazardRatio', 'PValue'])
    else:
        hazard_df = pd.concat(hazard_rows, ignore_index=True)

    return hazard_df, cox_models

MAIN PIPELINE

In [9]:
def main():
    log.info("BOXBOX - Phase 4: Survival Analysis")
    log.info("-" * 50)

    df = load_data()

    stint_df = build_stint_dataset(df)

    survival_curves_df, km_models = fit_kaplan_meier(stint_df)
    hazard_df, cox_models = fit_cox_models(stint_df)

    #SAVE SURVIVAL CURVES
    curved_path = os.path.join(BASE, 'data', 'processed', 'survival_curves.csv')
    survival_curves_df.to_csv(curved_path, index=False)
    log.info(f"\nSaved: {curved_path} ({len(survival_curves_df)} rows)")

    #SAVE COX HAZARD RATIOS
    hazard_path = os.path.join(BASE, 'data', 'outputs', 'cox_hazard_ratios.csv')
    hazard_df.to_csv(hazard_path, index=False)
    log.info(f"Saved: {hazard_path} ({len(hazard_df)} rows)")

    #SAVING STINT-LEVEL DATASET (USEFUL FOR DEBUGGING/DASHBOARD)
    stint_path = os.path.join(BASE, 'data', 'processed', 'stint_survival_data.csv')
    stint_df.to_csv(stint_path, index=False)
    log.info(f"Saved: {stint_path} ({len(stint_df)} rows)")

    #SAVE BOTH MODEL TYPES BUNDLED TOGETHER
    models_path = os.path.join(BASE, 'models', 'survival_models.pkl')
    joblib.dump({
        'kaplan_meier': km_models,
        'cox': cox_models
    }, models_path)
    log.info(f"Saved: {models_path}"
            f"({len(km_models)} KM curves, {len(cox_models)} Cox models)")

    #SUMMARY
    log.info("Phase 4 Summary")
    log.info("-" * 50)

    log.info(f"\nTotal stints analyzed: {len(stint_df)}")
    log.info(f"Kaplan-Meier curves fitted: {len(km_models)}")
    log.info(f"Cox models fitted: {len(cox_models)} (one per circuit)")

    log.info("\nTop 5 highest-risk factors across all ciruits "
            f"(highest hazard ratio):")
    top_hazards = hazard_df.sort_values('HazardRatio', ascending=False).head(5)
    log.info(top_hazards.to_string(index=False))

if __name__ == '__main__':
    main()

2026-07-28 07:20:19,729 - INFO - BOXBOX - Phase 4: Survival Analysis
2026-07-28 07:20:19,731 - INFO - --------------------------------------------------
2026-07-28 07:20:19,880 - INFO - Loaded engineered_laps.csv: 20887 rows
2026-07-28 07:20:19,881 - INFO - BUILDING STINT-LEVEL SIRVIVAL DATASET
2026-07-28 07:20:19,882 - INFO - --------------------------------------------------
2026-07-28 07:20:20,382 - INFO - Stint-level dataset built: 1048 stints
2026-07-28 07:20:20,383 - INFO -  Failure events: 230
2026-07-28 07:20:20,384 - INFO -  Censored (pitted before failing):818
2026-07-28 07:20:20,385 - INFO - FITTING KAPLAN-MEIER SURVIVAL CURVES
2026-07-28 07:20:20,385 - INFO - --------------------------------------------------
2026-07-28 07:20:20,398 - INFO -  Abu Dhabi / HARD: 24 stints | median survival = inf laps
2026-07-28 07:20:20,408 - INFO -  Abu Dhabi / MEDIUM: 16 stints | median survival = inf laps
2026-07-28 07:20:20,408 - WARNING -  Skipping Abu Dhabi / SOFT - only 1 stints (need 